In [ ]:
import pandas as pd
import itertools
import os
import json
import boto3

print("Starting Task 2: Building Master Action Registry...")

def get_use_case_id(default: str = "pl-aip-uplift") -> str:
    value = os.getenv("USE_CASE_ID", default).strip()
    if not value:
        raise ValueError("USE_CASE_ID must be non-empty.")
    return value

USE_CASE_ID = get_use_case_id("pl-aip-uplift")

# 1. Load the Action Banks JSON directly from S3
s3 = boto3.client('s3')
bucket_name = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
key = f'config/action_banks_{USE_CASE_ID}_V2_FEATURE_BASED.json'

try:
    response = s3.get_object(Bucket=bucket_name, Key=key)
    action_config = json.loads(response['Body'].read().decode('utf-8'))
    print(f"Successfully loaded JSON from s3://{bucket_name}/{key}")
except Exception as e:
    print(f"Error loading from S3: {e}")
    print("Attempting to load from local directory fallback...")
    try:
        local_name = f"action_banks_{USE_CASE_ID}_V2_FEATURE_BASED.json"
        with open(local_name, "r", encoding="utf-8") as f:
            action_config = json.load(f)
        print("Successfully loaded JSON from local directory.")
    except Exception as local_e:
         raise RuntimeError(f"Local fallback failed: {local_e}") from local_e

USE_CASE_ID = action_config.get("use_case_id", USE_CASE_ID)
CURRENT_VERSION = action_config.get("version", "2.0")
print(f"Use Case: {USE_CASE_ID} | Version: {CURRENT_VERSION}")


In [ ]:
# 2. Dynamically extract Dimension IDs from the JSON
print("Starting dynamic extraction of Dimension IDs...")

expected_keys = ["frequency", "channel", "day", "offer", "creative", "time"]
dimension_ids = {k: [] for k in expected_keys}

for dim_name, dim_data in action_config.get("dimensions", {}).items():
    dim_ids = set()
    for feature_name, feature_data in dim_data.get("features", {}).items():
        
        # Check values array for direct IDs (e.g., "CR001")
        for val in feature_data.get("values", []):
            val_str = str(val)
            if len(val_str) == 5 and val_str[:2].isalpha() and val_str[2:].isdigit():
                dim_ids.add(val_str)
        
        # Check labels dict for prefixed IDs (e.g., "FR001 - 1x per week")
        for label in feature_data.get("labels", {}).values():
            potential_id = label.split(" - ")[0].strip()
            if len(potential_id) == 5 and potential_id[:2].isalpha() and potential_id[2:].isdigit():
                dim_ids.add(potential_id)
                
    if dim_ids:
        # Map back to our dictionary and sort alphanumerically
        if dim_name in dimension_ids:
            dimension_ids[dim_name] = sorted(list(dim_ids))
        else:
            dimension_ids[dim_name] = sorted(list(dim_ids))

# Clean up any missing dimensions and finalize keys
dimension_ids = {k: v for k, v in dimension_ids.items() if v}
keys = list(dimension_ids.keys())

print("Extraction complete. Found the following IDs:")
for k, v in dimension_ids.items():
    print(f"  {k}: {v}")


In [ ]:
# 3. Generate ALL combinations using Cartesian product
print("Generating Cartesian product of all dimensions...")

value_lists = [dimension_ids[k] for k in keys]
all_combinations = list(itertools.product(*value_lists))

# Create the new DataFrame matching your column specifications
new_config_df = pd.DataFrame(all_combinations, columns=keys)
new_config_df.insert(0, 'use_case_id', USE_CASE_ID)

# Formatting version to ensure it has 'v' prefix (e.g., v2.0)
version_str = str(CURRENT_VERSION)
if not version_str.startswith('v'):
    version_str = f"v{version_str}"
new_config_df.insert(1, 'config_version', version_str)
new_config_df.insert(0, 'action_id', range(len(new_config_df)))

# Create a unique signature string for the upsert comparison
new_config_df['combo_signature'] = new_config_df[keys].agg('-'.join, axis=1)

print(f"Success! Generated {len(new_config_df)} possible combinations for version {version_str}.")
print("\nPreview of new generated combinations:")
print(new_config_df[['action_id', 'use_case_id', 'config_version'] + keys].head(3).to_string(index=False))

# Save the local parquet before preview/upload cells consume it.
output_file = f"action_library_{USE_CASE_ID}.parquet"
new_config_df.to_parquet(output_file, index=False)
print(f"Saved action library to {output_file}")


In [ ]:
# 5. Display the final output preview
print("\nPREVIEW: MASTER ACTION REGISTRY (Formatted Output)")
print("==========================================================================================")

# Load the file we just saved to ensure data integrity
final_registry_df = pd.read_parquet(f"action_library_{USE_CASE_ID}.parquet")

# Selecting columns to match your required layout
display_cols = ['action_id', 'use_case_id', 'config_version', 'frequency', 'channel', 'day', 'offer', 'creative', 'time']

# Displaying a clean view of the first few rows
print(final_registry_df[display_cols].head(10).to_string(index=False))

print("\n...")

# Displaying the final few rows to confirm the total count
print(final_registry_df[display_cols].tail(5).to_string(index=False))
print("==========================================================================================")
print(f"Total Rows: {len(final_registry_df)} | Unique Action IDs: 0 to {final_registry_df['action_id'].max()}")


In [ ]:
# Cell 6: Upload Master Action Registry to S3
import boto3
from botocore.exceptions import ClientError

# 1. Define S3 Path
s3_bucket = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
s3_folder = 'action_library/'
file_name = f"action_library_{USE_CASE_ID}.parquet"
s3_key = f"{s3_folder}{file_name}"

print(f"Preparing to upload {file_name} to s3://{s3_bucket}/{s3_key}...")

# 2. Initialize S3 Client
s3_client = boto3.client('s3')

try:
    # 3. Upload the local file
    s3_client.upload_file(file_name, s3_bucket, s3_key)
    print(f"Successfully uploaded to S3!")
    
    # 4. Verify the upload
    response = s3_client.head_object(Bucket=s3_bucket, Key=s3_key)
    print(f"Verification: File size in S3 is {response['ContentLength']} bytes.")
    print(f"Last Modified: {response['LastModified']}")

except ClientError as e:
    print(f"Error uploading to S3: {e}")
    raise
except FileNotFoundError:
    print(f"Error: Local file {file_name} not found. Please run Cell 4 again.")
    raise
